# 02 — CNN Model (Keras/TensorFlow)
Train a compact CNN on the Kaggle Digit Recognizer (MNIST) data and create a submission.

In [ ]:
# imports
%pip -q install tensorflow==2.* seaborn scikit-learn --disable-pip-version-check

In [ ]:
import os, numpy as np, pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from src.utils import seed_everything, load_train_test, as_images, plot_samples, save_submission
seed_everything(42)

DATA_DIR = "data"
SUB_PATH = "submissions/submission_cnn_v1.csv"
MODEL_PATH = "models/cnn_v1.keras"
os.makedirs("models", exist_ok=True)
os.makedirs("submissions", exist_ok=True)

In [ ]:
# Load data
X, y, X_test = load_train_test(DATA_DIR)
X_img = as_images(X)
X_test_img = as_images(X_test)
plot_samples(X, y, n=16, title="Training samples")

In [ ]:
# Build model
def build_model():
    model = keras.Sequential([
        layers.Conv2D(32, 3, activation="relu", input_shape=(28,28,1)),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.25),
        layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_model()
model.summary()

In [ ]:
# Augmentation + training with validation split
datagen = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=8, width_shift_range=0.1, height_shift_range=0.1, zoom_range=0.1, validation_split=0.1
)
datagen.fit(X_img)
train_flow = datagen.flow(X_img, y, batch_size=128, subset="training")
val_flow = datagen.flow(X_img, y, batch_size=128, subset="validation")

callbacks = [keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_accuracy")]
history = model.fit(train_flow, epochs=10, validation_data=val_flow, callbacks=callbacks, verbose=2)

In [ ]:
# Predict & save submission
pred = np.argmax(model.predict(X_test_img, verbose=0), axis=1)
save_submission(pred, SUB_PATH)
model.save(MODEL_PATH)
print(f"Saved: {SUB_PATH}\nSaved model: {MODEL_PATH}")